# Sketch 3.3 - standalone, portable notebook

Self-contained extract of **sketch_3_3_positional** from the combined *9 sketches*
notebook. The sketch code is unchanged; the shared setup it needs
(imports, data loading) is included below. Chart data is **inlined** (no external
`altair-data` files) so it renders on Linux, macOS and Windows alike.

## Shared setup *(unchanged)*

In [ ]:
import altair as alt
import pandas as pd
import numpy as np
import geopandas as gpd  # conda install -c conda-forge geopandas
import json

# Inline the chart data so every HTML output is SELF-CONTAINED and renders on
# all platforms (incl. macOS): external altair-data-*.json URL files are not
# fetched by many notebook frontends, which left the charts blank on Mac. The
# search sketches (1.13 / 2.4 / 2.6) restrict their name universe below so the
# inlined tables stay small.
alt.data_transformers.enable('default')
alt.data_transformers.disable_max_rows()
pass

In [ ]:
# Memory-lean load: another heavy job is running on this machine (~2.5 GB free),
# so we read the CSV with categorical dtypes (15k name strings stored once), derive
# the numeric year from the 122 'annais' categories instead of 3.5M strings, then
# relax categories back to plain objects (the strings stay shared) so every later
# groupby keeps its normal semantics.
raw = pd.read_csv('dpt2020.csv', sep=';',
                  dtype={'sexe': 'int8', 'preusuel': 'category',
                         'annais': 'category', 'dpt': 'category', 'nombre': 'int32'})
cat_years = pd.to_numeric(pd.Index(raw['annais'].cat.categories), errors='coerce').astype('float32')
raw['year'] = np.asarray(cat_years)[raw['annais'].cat.codes]
raw.drop(columns=['annais'], inplace=True)
raw['decade'] = (raw['year'] // 10 * 10)
raw['preusuel'] = raw['preusuel'].astype(object)  # shared strings, plain groupbys
raw['dpt'] = raw['dpt'].astype(object)

records = raw[raw.dpt != 'XX'].copy()
named = records[records.preusuel != '_PRENOMS_RARES'].copy()

dept_total = records.groupby('dpt', as_index=False)['nombre'].sum().rename(
    columns={'nombre': 'dept_total'})

ALL_NAMES = sorted(named.preusuel.unique().tolist())  # full dropdown option list
DECADES = list(range(1900, 2021, 10))
# Shared period options for charts that need an all-years aggregate (2.4).
DECADE_OPTS = [0] + DECADES
DECADE_LABELS = ['All years'] + [str(d) for d in DECADES]
SEX_OPTIONS = ['Mixed', 'Boys', 'Girls']
SEX_LABELS = ['mixed', 'boys', 'girls']

import gc
del raw
gc.collect()

print(f'{len(named):,} named rows; {len(ALL_NAMES):,} distinct names.')
named.sample(3)


## The sketch

### Sketch 3.3 - Names by gender lean (year slider, all 121 years)

X is a logit of the male share (centre = unisex, Male on the left); colour is a
3-class gender label; size = births (√). Only genuinely co-ed names (each sex ≥ 5%)
are labelled. The year slider covers 1900-2020.


In [ ]:
ya = (named.dropna(subset=['year'])
      .groupby(['year', 'preusuel', 'sexe'])['nombre'].sum()
      .unstack(fill_value=0).rename(columns={1: 'male', 2: 'female'}).reset_index())
ya['total'] = ya['male'] + ya['female']
ya['rank'] = ya.groupby('year')['total'].rank(ascending=False, method='first')
ya = ya[ya['rank'] <= 1000].copy()
ya['male_share'] = ya['male'] / ya['total']
s = ya['male_share'].clip(0.01, 0.99)
ya['lean'] = -np.log(s / (1 - s))  # negative => male on the LEFT (sketch orientation)
ya['cls'] = np.where(ya.male_share >= 0.6, 'Mostly male',
                     np.where(ya.male_share <= 0.4, 'Mostly female', 'Unisex'))
ya['mixed'] = ya['male_share'].between(0.05, 0.95)  # given to both sexes >=5% -> co-ed (labelled)
ya['year'] = ya['year'].astype(int)

n_pts = alt.param(name='n_pts', value=300,
                  bind=alt.binding_range(min=50, max=1000, step=50, name='Top N names plotted  '))
year_sel = alt.selection_point(fields=['year'], value=[{'year': 2000}],
                               bind=alt.binding_range(min=1900, max=2020, step=1, name='Year  '))
pts = alt.Chart(ya).mark_circle(opacity=0.8, stroke='#777', strokeWidth=0.3).encode(
    x=alt.X('lean:Q', title='<- more MALE      gender lean      more FEMALE ->',
            axis=alt.Axis(labels=False, ticks=False, grid=False)),
    y=alt.Y('total:Q', title='Births this year (log)', scale=alt.Scale(type='log')),
    size=alt.Size('total:Q', scale=alt.Scale(type='sqrt', range=[20, 1100]), title='Births'),
    color=alt.Color('cls:N', title='Gender class',
                    scale=alt.Scale(domain=['Mostly male', 'Unisex', 'Mostly female'],
                                    range=['#4c78a8', '#f58518', '#e45756'])),
    tooltip=[alt.Tooltip('preusuel:N', title='Name'), alt.Tooltip('year:Q', title='Year', format='d'),
             alt.Tooltip('total:Q', title='Births', format=','),
             alt.Tooltip('male_share:Q', title='Male share', format='.0%')],
).transform_filter(year_sel).transform_filter('datum.rank <= n_pts')
mid = alt.Chart(pd.DataFrame({'x': [0]})).mark_rule(color='#888', strokeDash=[4, 4]).encode(x='x:Q')
txt = alt.Chart(ya).mark_text(dy=-12, fontSize=9, fontWeight='bold').encode(
    x='lean:Q', y='total:Q', text='preusuel:N').transform_filter(
    year_sel).transform_filter('datum.rank <= n_pts').transform_filter(alt.datum.mixed == True)
sketch_3_3 = (mid + pts + txt).add_params(year_sel, n_pts).properties(
    width=760, height=470,
    title=alt.TitleParams(
        text='3.3 - Names by gender lean and popularity (drag the year)',
        subtitle='Almost every French name is strongly gendered (two clusters); labelled names are the rare co-ed ones.',
        anchor='start'))

sketch_3_3.save('sketch_3_3_positional.png', ppi=120)
sketch_3_3